# 1. Project Introduction: DS_Day01_19 Restaurant Staffing

This project examines hourly order demand, staffing, preparation time, and queue pressure by outlet, day type, and holiday flag. **The included data is synthetic and reproducible**, generated for this exercise because no source dataset was present. Its results are illustrative, not operational evidence.

## 2. Import Libraries

In [1]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd

candidate_root = Path.cwd()
PROJECT_ROOT = candidate_root if (candidate_root / 'run_analysis.py').exists() else candidate_root.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from run_analysis import (
    DATA_PATH, OUTPUT_DIR, MODEL_DIR, REQUIRED_COLUMNS, add_orders_per_employee,
    clean_dataset, classify_staffing, create_action_plan, create_visualizations,
    descriptive_statistics, ensure_directories, generate_insights, inspect_data,
    load_source_data, outlet_comparison, peak_demand_periods,
    staffing_queue_relationship, train_random_forest
)

## 3. Load Dataset

In [2]:
ensure_directories()
source_data, synthetic_data = load_source_data()
print('Source:', 'fixed-seed synthetic sample' if synthetic_data else DATA_PATH)
print('Shape:', source_data.shape)
display(source_data.head())

Source: fixed-seed synthetic sample
Shape: (5760, 9)


,Date,Outlet_ID,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Day_Type,Holiday
0,2025-01-01,OUTLET_A,7,9,5.51,0,2,Weekday,False
1,2025-01-01,OUTLET_A,8,15,4.16,0,6,Weekday,False
2,2025-01-01,OUTLET_A,9,9,4.77,1,2,Weekday,False
3,2025-01-01,OUTLET_A,10,12,5.98,4,2,Weekday,False
4,2025-01-01,OUTLET_A,11,12,5.13,0,3,Weekday,False


## 4. Data Inspection

In [3]:
inspection = inspect_data(source_data)
print('Columns:', inspection['columns'])
print('Shape:', inspection['shape'])
print('Exact duplicate rows:', inspection['duplicate_rows'])
display(inspection['first_five_rows'])
display(inspection['data_types'].rename('dtype').to_frame())
display(inspection['missing_values'].rename('missing').to_frame())

Columns: ['Date', 'Outlet_ID', 'Hour', 'Order_Volume', 'Average_Preparation_Time', 'Queue_Length', 'Staff_Count', 'Day_Type', 'Holiday']
Shape: (5760, 9)
Exact duplicate rows: 0


,Date,Outlet_ID,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Day_Type,Holiday
0,2025-01-01,OUTLET_A,7,9,5.51,0,2,Weekday,False
1,2025-01-01,OUTLET_A,8,15,4.16,0,6,Weekday,False
2,2025-01-01,OUTLET_A,9,9,4.77,1,2,Weekday,False
3,2025-01-01,OUTLET_A,10,12,5.98,4,2,Weekday,False
4,2025-01-01,OUTLET_A,11,12,5.13,0,3,Weekday,False


,dtype
Date,object
Outlet_ID,object
Hour,int64
Order_Volume,int64
Average_Preparation_Time,float64
Queue_Length,int64
Staff_Count,int64
Day_Type,object
Holiday,bool


,missing
Date,0
Outlet_ID,0
Hour,0
Order_Volume,0
Average_Preparation_Time,0
Queue_Length,0
Staff_Count,0
Day_Type,0
Holiday,0


## 5. Data Cleaning

In [4]:
cleaned, cleaning_report = clean_dataset(source_data)
data = add_orders_per_employee(cleaned)
data.to_csv(OUTPUT_DIR / 'cleaned_dataset.csv', index=False)
print('Cleaning report:', cleaning_report)
print('Numeric gaps use outlet-hour medians, then an overall median. Invalid date/outlet/hour rows are excluded; high values are retained.')
display(data.head())

Cleaning report: {'input_rows': 5760, 'duplicates_removed': 0, 'rows_missing_valid_date_outlet_or_hour': 0, 'output_rows': 5760, 'missing_values_after_cleaning': 0}
Numeric gaps use outlet-hour medians, then an overall median. Invalid date/outlet/hour rows are excluded; high values are retained.


,Date,Outlet_ID,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Day_Type,Holiday,Orders_Per_Employee
0,2025-01-01,OUTLET_A,7,9,5.51,0,2,Weekday,False,4.5
1,2025-01-01,OUTLET_A,8,15,4.16,0,6,Weekday,False,2.5
2,2025-01-01,OUTLET_A,9,9,4.77,1,2,Weekday,False,4.5
3,2025-01-01,OUTLET_A,10,12,5.98,4,2,Weekday,False,6.0
4,2025-01-01,OUTLET_A,11,12,5.13,0,3,Weekday,False,4.0


## 6. Data Validation

In [5]:
assert list(cleaned.columns) == REQUIRED_COLUMNS
assert cleaned['Date'].notna().all()
assert cleaned['Outlet_ID'].notna().all()
assert cleaned['Hour'].between(0, 23).all()
assert (cleaned[['Order_Volume', 'Average_Preparation_Time', 'Queue_Length', 'Staff_Count']] >= 0).all().all()
assert set(cleaned['Day_Type'].unique()).issubset({'Weekday', 'Weekend'})
assert set(cleaned['Holiday'].unique()).issubset({True, False})
assert cleaned.isna().sum().sum() == 0
print('Required columns, hours, non-negative measures, categories, and missing-value checks passed.')

Required columns, hours, non-negative measures, categories, and missing-value checks passed.


## 7. Descriptive Statistics

In [6]:
statistics = descriptive_statistics(data)
statistics.to_csv(OUTPUT_DIR / 'descriptive_statistics.csv', index=False)
display(statistics)

,Variable,count,mean,median,std,min,max
0,Order_Volume,5760.0,18.876,18.00,9.300,1.00,57.00
1,Average_Preparation_Time,5760.0,5.175,5.14,0.923,1.95,9.37
2,Queue_Length,5760.0,0.975,0.00,1.337,0.00,9.00
3,Staff_Count,5760.0,4.262,4.00,1.659,1.00,10.00


## 8. Exploratory Data Analysis

In [7]:
hourly = data.groupby('Hour', as_index=False).agg(
    Order_Volume=('Order_Volume', 'mean'),
    Average_Preparation_Time=('Average_Preparation_Time', 'mean'),
    Queue_Length=('Queue_Length', 'mean'),
    Staff_Count=('Staff_Count', 'mean'),
    Orders_Per_Employee=('Orders_Per_Employee', 'mean')
)
by_day_type = data.groupby('Day_Type')[['Order_Volume', 'Average_Preparation_Time', 'Queue_Length', 'Staff_Count', 'Orders_Per_Employee']].mean().round(3)
by_holiday = data.groupby('Holiday')[['Order_Volume', 'Average_Preparation_Time', 'Queue_Length', 'Staff_Count', 'Orders_Per_Employee']].mean().round(3)
print('Hourly averages:')
display(hourly.round(3))
print('Weekday/weekend comparison:')
display(by_day_type)
print('Holiday/non-holiday comparison:')
display(by_holiday)

Hourly averages:


,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Orders_Per_Employee
0,7,7.233,4.703,0.408,3.017,2.903
1,8,10.414,4.863,0.647,3.381,3.630
2,9,9.539,4.945,0.717,3.119,3.722
3,10,9.678,4.913,0.528,3.356,3.372
4,11,14.419,5.115,0.939,3.669,4.560
5,12,24.167,5.361,1.128,4.889,5.308
6,13,28.747,5.408,1.428,5.311,5.703
7,14,23.125,5.260,1.122,4.733,5.190
8,15,17.039,5.175,0.953,4.006,4.751
9,16,14.372,5.094,0.800,3.744,4.324


Weekday/weekend comparison:


,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Orders_Per_Employee
Day_Type,,,,,
Weekday,18.088,5.202,1.030,4.041,4.788
Weekend,20.869,5.106,0.837,4.823,4.439


Holiday/non-holiday comparison:


,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Orders_Per_Employee
Holiday,,,,,
False,18.791,5.171,0.971,4.257,4.673
True,21.333,5.277,1.099,4.422,5.148


## 9. Visualisations

In [8]:
figure_paths = create_visualizations(data)
print('Generated figures:')
for figure_path in figure_paths:
    print(figure_path.relative_to(PROJECT_ROOT))
print('Required charts: order volume by hour, staff vs queue, orders per employee, and outlet comparison.')

Generated figures:
outputs\figures\order_volume_by_hour.png
outputs\figures\staff_count_vs_queue_length.png
outputs\figures\orders_per_employee.png
outputs\figures\outlet_comparison.png
outputs\figures\preparation_time_by_hour.png
outputs\figures\queue_length_by_hour.png
outputs\figures\staff_count_by_hour.png
Required charts: order volume by hour, staff vs queue, orders per employee, and outlet comparison.


## 10. Peak Demand Analysis

In [9]:
peaks = peak_demand_periods(data)
peaks.to_csv(OUTPUT_DIR / 'peak_demand_periods.csv', index=False)
display(peaks.head(20))
print('Peak groups are those at or above the 90th percentile of group-average order volume within each requested grouping.')

F:\DataScience_project_2\DS_Day01_19_Restaurant_Staffing\run_analysis.py:287: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(peak_tables, ignore_index=True).sort_values(


,Analysis_Level,Hour,Outlet_ID,Day_Type,Holiday,Order_Volume,Staff_Count,Queue_Length,Average_Preparation_Time,Orders_Per_Employee,Observation_Count
0,Day type and hour,19,<NA>,Weekend,<NA>,34.647059,6.421569,1.186275,5.369314,5.567651,102
1,Day type and hour,20,<NA>,Weekend,<NA>,31.813725,6.196078,0.941176,5.290588,5.262352,102
2,Day type and hour,13,<NA>,Weekend,<NA>,30.892157,5.970588,1.196078,5.274412,5.346106,102
3,Day type and hour,18,<NA>,Weekend,<NA>,30.862745,5.921569,1.127451,5.349412,5.392456,102
4,Holiday and hour,19,<NA>,NaN,True,36.500000,6.000000,1.166667,5.240833,6.172421,12
5,Holiday and hour,13,<NA>,NaN,True,33.000000,5.583333,1.333333,5.264167,6.072421,12
6,Holiday and hour,20,<NA>,NaN,True,31.750000,6.166667,0.916667,5.236667,5.259722,12
7,Holiday and hour,19,<NA>,NaN,False,31.353448,5.715517,1.408046,5.508247,5.808264,348
8,Hour,19,<NA>,NaN,<NA>,31.525000,5.725000,1.400000,5.499333,5.820402,360
9,Hour,13,<NA>,NaN,<NA>,28.747222,5.311111,1.427778,5.408194,5.703016,360


Peak groups are those at or above the 90th percentile of group-average order volume within each requested grouping.


## 11. Orders per Employee

In [10]:
workload_by_hour = data.groupby('Hour', as_index=False)['Orders_Per_Employee'].mean()
workload_by_outlet = data.groupby('Outlet_ID', as_index=False)['Orders_Per_Employee'].mean()
workload_by_day_type = data.groupby('Day_Type', as_index=False)['Orders_Per_Employee'].mean()
print('Formula: Order_Volume / Staff_Count; zero staff yields an undefined ratio.')
display(workload_by_hour.round(3))
display(workload_by_outlet.round(3))
display(workload_by_day_type.round(3))

Formula: Order_Volume / Staff_Count; zero staff yields an undefined ratio.


,Hour,Orders_Per_Employee
0,7,2.903
1,8,3.630
2,9,3.722
3,10,3.372
4,11,4.560
5,12,5.308
6,13,5.703
7,14,5.190
8,15,4.751
9,16,4.324


,Outlet_ID,Orders_Per_Employee
0,OUTLET_A,5.013
1,OUTLET_B,4.665
2,OUTLET_C,4.389


,Day_Type,Orders_Per_Employee
0,Weekday,4.788
1,Weekend,4.439


## 12. Understaffing and Overstaffing

In [11]:
staffing, staffing_thresholds = classify_staffing(data)
staffing.to_csv(OUTPUT_DIR / 'staffing_analysis.csv', index=False)
print('Data-derived thresholds:', staffing_thresholds)
display(staffing['Staffing_Status'].value_counts().rename_axis('Staffing_Status').to_frame('Periods'))
display(staffing.head())
print('Understaffed: workload above its data Q75, or queue above Q75 with preparation above its median. Overstaffed: workload below Q25 with queue and preparation below their medians. Otherwise balanced.')

Data-derived thresholds: {'orders_per_employee_q25': 3.2857142857142856, 'orders_per_employee_q75': 5.6, 'queue_length_median': 0.0, 'queue_length_q75': 2.0, 'preparation_time_median': 5.14, 'preparation_time_q75': 5.762499999999999}


,Periods
Staffing_Status,
Balanced,111
Understaffed,31
Overstaffed,2


,Outlet_ID,Hour,Day_Type,Holiday,Order_Volume,Staff_Count,Queue_Length,Average_Preparation_Time,Orders_Per_Employee,Observation_Count,Staffing_Status
0,OUTLET_A,7,Weekday,False,8.170732,2.963415,0.487805,4.777561,3.312805,82,Balanced
1,OUTLET_A,7,Weekday,True,10.250000,2.500000,2.250000,5.042500,5.416667,4,Balanced
2,OUTLET_A,7,Weekend,False,9.176471,3.764706,0.235294,4.591765,2.762745,34,Balanced
3,OUTLET_A,8,Weekday,False,11.573171,3.439024,0.695122,4.881463,3.843902,82,Balanced
4,OUTLET_A,8,Weekday,True,13.250000,2.500000,2.750000,5.917500,6.625000,4,Understaffed


Understaffed: workload above its data Q75, or queue above Q75 with preparation above its median. Overstaffed: workload below Q25 with queue and preparation below their medians. Otherwise balanced.


## 13. Staffing vs Queue Length

In [12]:
correlations, queue_by_staff = staffing_queue_relationship(data)
correlations.to_csv(OUTPUT_DIR / 'staff_queue_correlations.csv')
queue_by_staff.to_csv(OUTPUT_DIR / 'staff_queue_grouped_statistics.csv', index=False)
print('Pearson correlations:')
display(correlations)
print('Grouped statistics by staff count:')
display(queue_by_staff)
print('Queue length is a queue-pressure proxy, not a direct wait-time measure. Correlation does not establish causation.')

Pearson correlations:


,Staff_Count,Queue_Length,Order_Volume,Average_Preparation_Time
Staff_Count,1.000,-0.292,0.658,-0.175
Queue_Length,-0.292,1.000,0.220,0.458
Order_Volume,0.658,0.220,1.000,0.247
Average_Preparation_Time,-0.175,0.458,0.247,1.000


Grouped statistics by staff count:


,Staff_Count,Queue_Length,Order_Volume,Average_Preparation_Time,Observation_Count
0,1,2.975610,8.369919,6.063333,246
1,2,1.762712,12.667797,5.546593,590
2,3,0.992681,13.599268,5.148097,1093
3,4,0.777613,17.294292,5.088732,1349
4,5,0.695286,20.457071,5.037677,1188
5,6,0.698138,25.452128,5.042327,752
6,7,0.629428,30.277929,5.152180,367
7,8,0.714286,35.489796,5.057551,147
8,9,0.884615,43.307692,5.002692,26
9,10,0.500000,49.500000,4.920000,2


Queue length is a queue-pressure proxy, not a direct wait-time measure. Correlation does not establish causation.


## 14. Outlet Comparison

In [13]:
outlets = outlet_comparison(data)
outlets.to_csv(OUTPUT_DIR / 'outlet_comparison.csv', index=False)
display(outlets)

,Outlet_ID,Order_Volume,Staff_Count,Queue_Length,Average_Preparation_Time,Orders_Per_Employee,Observation_Count
0,OUTLET_A,22.091,4.599,1.099,5.252,5.013,1920
1,OUTLET_B,18.803,4.282,0.958,5.181,4.665,1920
2,OUTLET_C,15.734,3.906,0.868,5.091,4.389,1920


## 15. Random Forest

In [14]:
model, model_metrics, feature_importance, actual_queue, predicted_queue = train_random_forest(data)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_DIR / 'random_forest_model.pkl')
print('Target: Queue_Length. Inputs: Hour, Outlet_ID, Order_Volume, Average_Preparation_Time, Staff_Count, Day_Type, Holiday.')
print('The target is excluded from features. Categorical inputs are one-hot encoded. The chronological split tests later dates.')
display(model_metrics)

Target: Queue_Length. Inputs: Hour, Outlet_ID, Order_Volume, Average_Preparation_Time, Staff_Count, Day_Type, Holiday.
The target is excluded from features. Categorical inputs are one-hot encoded. The chronological split tests later dates.


,Target,Split,Train_Rows,Test_Rows,MAE,MSE,RMSE,R2
0,Queue_Length,Chronological 80/20 by unique date,4608,1152,0.6684,0.811,0.9006,0.5275


## 16. Model Evaluation

In [15]:
evaluation = model_metrics.copy()
evaluation['Top_Features'] = '; '.join(
    f'{row.Feature} ({row.Importance:.3f})' for row in feature_importance.head(5).itertuples()
)
evaluation.to_csv(OUTPUT_DIR / 'model_evaluation.csv', index=False)
feature_importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
figure_paths = create_visualizations(data, actual_queue, predicted_queue)
print('Metrics:')
display(evaluation)
print('Feature importance is predictive model attribution, not evidence of causation.')
display(feature_importance.head(10))
print('Saved actual-vs-predicted figure:', OUTPUT_DIR / 'figures' / 'actual_vs_predicted_queue.png')

Metrics:


,Target,Split,Train_Rows,Test_Rows,MAE,MSE,RMSE,R2,Top_Features
0,Queue_Length,Chronological 80/20 by unique date,4608,1152,0.6684,0.811,0.9006,0.5275,Average_Preparation_Time (0.363); Staff_Count ...


Feature importance is predictive model attribution, not evidence of causation.


,Feature,Importance
0,Average_Preparation_Time,0.36299
1,Staff_Count,0.25540
2,Order_Volume,0.24118
3,Hour,0.08091
4,Outlet_ID_OUTLET_B,0.01366
5,Outlet_ID_OUTLET_C,0.01328
6,Outlet_ID_OUTLET_A,0.01326
7,Day_Type_Weekday,0.00855
8,Day_Type_Weekend,0.00846
9,Holiday_True,0.00116


Saved actual-vs-predicted figure: F:\DataScience_project_2\DS_Day01_19_Restaurant_Staffing\outputs\figures\actual_vs_predicted_queue.png


## 17. Key Insights

In [16]:
insights = generate_insights(data, peaks, staffing, outlets, correlations)
provenance = 'Fixed-seed synthetic teaching data; not observed restaurant data.' if synthetic_data else 'Dataset read from the project CSV.'
with (OUTPUT_DIR / 'key_insights.txt').open('w', encoding='utf-8') as insight_file:
    insight_file.write(f'DATA PROVENANCE: {provenance}\n\n')
    for number, insight in enumerate(insights, start=1):
        insight_file.write(f'{number}. {insight}\n')
assert 5 <= len(insights) <= 7
print(provenance)
for number, insight in enumerate(insights, start=1):
    print(f'{number}. {insight}')

Fixed-seed synthetic teaching data; not observed restaurant data.
1. Mean demand peaks at 19:00, averaging 31.52 orders per outlet-hour.
2. Weekend observations have the higher mean demand (20.87 orders per outlet-hour).
3. The holiday group has the higher mean demand (21.33 orders per outlet-hour; holiday/non-holiday ratio 1.14).
4. OUTLET_A has the highest outlet-average queue length (1.10) and averages 4.60 staff.
5. The highest mean workload is at OUTLET_A at 13:00 (6.10 orders per employee); OUTLET_A has the highest overall outlet-average workload (5.01).
6. The data-derived classification marks 21.5% of outlet-hour-day-type-holiday periods understaffed and 1.4% overstaffed. The raw staff-count/queue-length Pearson correlation is -0.292; this is observational and does not establish causation.


## 18. Practical Action Plan

In [17]:
action_plan = create_action_plan(staffing)
action_plan.to_csv(OUTPUT_DIR / 'practical_action_plan.csv', index=False)
display(action_plan.head(24))
print('Recommendations use observed group summaries and the stated heuristic labels; review against local operating context before use.')

,Time_Period,Outlet_ID,Day_Type,Holiday,Demand_Situation,Order_Volume,Staff_Count,Orders_Per_Employee,Queue_Length,Queue_Situation,Staffing_Status,Recommended_Action,Observation_Count
0,07:00,OUTLET_A,Weekday,False,Low relative to observed periods,8.171,2.963,3.313,0.488,Low relative to observed periods,Balanced,Maintain current staffing and continue monitor...,82
1,08:00,OUTLET_A,Weekday,False,Low relative to observed periods,11.573,3.439,3.844,0.695,Middle range of observed periods,Balanced,Maintain current staffing and continue monitor...,82
2,09:00,OUTLET_A,Weekday,False,Low relative to observed periods,10.744,2.793,4.698,1.073,Middle range of observed periods,Balanced,Maintain current staffing and continue monitor...,82
3,10:00,OUTLET_A,Weekday,False,Low relative to observed periods,10.561,3.390,3.456,0.415,Low relative to observed periods,Balanced,Maintain current staffing and continue monitor...,82
4,11:00,OUTLET_A,Weekday,False,Middle range of observed periods,16.110,3.939,4.679,1.024,Middle range of observed periods,Balanced,Maintain current staffing and continue monitor...,82
5,12:00,OUTLET_A,Weekday,False,High relative to observed periods,26.915,5.098,5.552,1.146,Middle range of observed periods,Balanced,Maintain current staffing and continue monitor...,82
6,13:00,OUTLET_A,Weekday,False,High relative to observed periods,32.890,5.500,6.324,1.793,High relative to observed periods,Understaffed,Increase staffing during comparable periods an...,82
7,14:00,OUTLET_A,Weekday,False,Middle range of observed periods,25.439,4.817,5.600,1.415,High relative to observed periods,Balanced,Maintain current staffing and continue monitor...,82
8,15:00,OUTLET_A,Weekday,False,Middle range of observed periods,19.293,4.110,5.212,1.207,Middle range of observed periods,Balanced,Maintain current staffing and continue monitor...,82
9,16:00,OUTLET_A,Weekday,False,Middle range of observed periods,16.134,3.732,4.803,0.963,Middle range of observed periods,Balanced,Maintain current staffing and continue monitor...,82


Recommendations use observed group summaries and the stated heuristic labels; review against local operating context before use.


## 19. Final Conclusion

This workflow compares observed groups in the bundled dataset and estimates queue length using one Random Forest regressor. Since the bundled records are synthetic, calculated patterns and staffing recommendations demonstrate the method only. Queue length is not direct waiting time, and observational relationships do not support causal conclusions.

In [18]:
print(f'Rows analyzed: {len(data):,}')
print(f'Peak mean-demand hour: {int(data.groupby("Hour")["Order_Volume"].mean().idxmax()):02d}:00')
print(f'Random Forest queue-length MAE: {model_metrics.loc[0, "MAE"]:.4f}')
print(f'Figures generated: {len(list((OUTPUT_DIR / "figures").glob("*.png")))}')
print(f'All tables, insights, plan, figures, and model are saved under {PROJECT_ROOT}.')

Rows analyzed: 5,760
Peak mean-demand hour: 19:00
Random Forest queue-length MAE: 0.6684
Figures generated: 8
All tables, insights, plan, figures, and model are saved under F:\DataScience_project_2\DS_Day01_19_Restaurant_Staffing.
